In [1]:
import geopandas as gpd

gdf = gpd.read_file("DISTRICT_BOUNDARY.shp")
gdf.to_file("districts.geojson", driver="GeoJSON")

In [2]:
import geopandas as gpd
gdf = gpd.read_file("DISTRICT_BOUNDARY.shp")
gdf["geometry"] = gdf["geometry"].simplify(0.05)  # increase tolerance
gdf.to_file("DISTRICT_BOUNDARY_small.geojson", driver="GeoJSON")

In [3]:
import geopandas as gpd
import os

# Load the original shapefile
print("Loading shapefile...")
gdf = gpd.read_file("DISTRICT_BOUNDARY.shp")

print(f"Original shape count: {len(gdf)}")
print(f"Original CRS: {gdf.crs}")
print(f"Columns: {list(gdf.columns)}")

# Reproject to geographic CRS (needed for simplify to work properly)
gdf = gdf.to_crs(epsg=4326)

# Simplify geometry - increase tolerance to reduce size more aggressively
# 0.01 = moderate simplification, 0.05 = aggressive
gdf["geometry"] = gdf["geometry"].simplify(tolerance=0.01, preserve_topology=True)

# Drop any columns you don't need (keep only essential ones)
# Adjust this list based on your actual columns
cols_to_keep = ["DISTRICT", "STATE", "geometry"]  # edit as needed
cols_to_keep = [c for c in cols_to_keep if c in gdf.columns]
gdf = gdf[cols_to_keep]

# Remove null/invalid geometries
gdf = gdf[gdf["geometry"].notna()]
gdf = gdf[~gdf["geometry"].is_empty]

# Save as new shapefile
output_path = "DISTRICT_BOUNDARY_small.shp"
gdf.to_file(output_path)

# Check output file size
size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"\nSaved to: {output_path}")
print(f"New .shp file size: {size_mb:.2f} MB")

# If still too large, try more aggressive simplification
if size_mb > 20:
    print("\nStill too large, applying aggressive simplification...")
    gdf["geometry"] = gdf["geometry"].simplify(tolerance=0.05, preserve_topology=True)
    gdf.to_file("DISTRICT_BOUNDARY_smaller.shp")
    size_mb2 = os.path.getsize("DISTRICT_BOUNDARY_smaller.shp") / (1024 * 1024)
    print(f"Aggressive version size: {size_mb2:.2f} MB")

Loading shapefile...
Original shape count: 808
Original CRS: PROJCS["LCC_WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",24],PARAMETER["central_meridian",80],PARAMETER["standard_parallel_1",12.472944],PARAMETER["standard_parallel_2",35.172806],PARAMETER["false_easting",4000000],PARAMETER["false_northing",4000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Columns: ['OBJECTID', 'OBJECTID_1', 'STATE_UT', 'STATE_LGD', 'DISTRICT', 'DIST_LGD', 'REMARKS', 'Shape_Leng', 'Shape_Area', 'geometry']

Saved to: DISTRICT_BOUNDARY_small.shp
New .shp file size: 1.76 MB
